# 🧠 DES Hallucination Proxy: 3D Embedding Visualization Lab

This notebook provides an interactive 3D laboratory to visualize model disagreement and hallucinations in semantic space.

### 🔬 Objectives:
1. **Semantic Clustering**: Visualize how different model architectures (Llama, GPT, Qwen, etc.) cluster for the same question.
2. **Hallucination Detection**: Identify "scattered" response patterns that correlate with high DES scores.
3. **Ensemble Analysis**: See where the "Wisdom of the Crowd" succeeds and where it fails.

In [43]:
import pandas as pd
import numpy as np
import json
import plotly.express as px
import plotly.graph_objects as go
import umap
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded.")

✅ Libraries loaded.


## 1. Load Scored Results
We load the results generated by the DES pipeline (`data/results/scored_results.jsonl`).

In [44]:
DATA_PATH = "../data/results/scored_results.jsonl"

def load_results(path):
    data = []
    with open(path, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    return pd.DataFrame(data)

df = load_results(DATA_PATH)
print(f"Loaded {len(df)} questions.")
df.head(2)

Loaded 2017 questions.


,id,source,domain,question,correct_answers,choices,question_type,model_responses,correctness_flags,any_error,surface_DES,semantic_DES,DES,null_models
0,tqa_0000,triviaqa,general,What general name is given to a rotating star ...,[PULSAR],None,open,{'llama4-scout': {'model_alias': 'llama4-scout...,"{'llama-large': True, 'llama-small': True, 'll...",0,0.222222,0.126227,0.145426,[]
1,tqa_0001,triviaqa,general,"""Which Gilbert and Sullivan operetta is sub ti...",[RUDDIGORE],None,open,{'llama4-scout': {'model_alias': 'llama4-scout...,"{'llama-large': True, 'llama-small': False, 'l...",1,0.805556,0.568607,0.615997,[]


## 2. Interactive 3D Visualization Logic
The core function to visualize a single question.

In [45]:
model_embed = SentenceTransformer('all-MiniLM-L6-v2')

def visualize_question_3d(qid_index, show_truth_lines=True):
    row = df.iloc[qid_index]
    question = row['question']
    responses_dict = row.get('model_responses', {})
    des_score = row.get('DES', 0.0)
    
    ground_truth_list = row.get('correct_answers', ['N/A'])
    correct_answer = str(ground_truth_list[0]) if isinstance(ground_truth_list, list) else str(ground_truth_list)

    model_names = list(responses_dict.keys())
    texts = [str(resp_obj.get('answer', '')) for resp_obj in responses_dict.values()]
    
    # Add Ground Truth
    texts.append(correct_answer)
    model_names.append("GROUND_TRUTH")

    embeddings = model_embed.encode(texts)
    reducer = umap.UMAP(n_components=3, n_neighbors=2, min_dist=0.1, random_state=42)
    embed_3d = reducer.fit_transform(embeddings)

    plot_df = pd.DataFrame(embed_3d, columns=['x', 'y', 'z'])
    plot_df['Model'] = model_names
    plot_df['Response'] = texts
    plot_df['Type'] = ['Model' if m != 'GROUND_TRUTH' else 'Truth' for m in model_names]
    
    # Calculate Metrics
    truth_coords = plot_df[plot_df['Type'] == 'Truth'][['x', 'y', 'z']].values[0]
    model_coords = plot_df[plot_df['Type'] == 'Model'][['x', 'y', 'z']].values
    centroid = np.mean(model_coords, axis=0)
    centroid_dist = np.linalg.norm(centroid - truth_coords)
    avg_spread = np.mean([np.linalg.norm(c - centroid) for c in model_coords])

    fig_title = f"QID {qid_index}: {question[:60]}...<br>DES: {des_score:.4f} | Centroid Shift: {centroid_dist:.2f} | Spread: {avg_spread:.2f}"
    
    fig = px.scatter_3d(
        plot_df, x='x', y='y', z='z', 
        color='Model', 
        symbol='Type',
        hover_data=['Response'],
        title=fig_title,
        labels={'x': 'UMAP 1', 'y': 'UMAP 2', 'z': 'UMAP 3'}
    )
    fig.update_traces(marker=dict(size=8), selector=dict(mode='markers'))

    # Add Truth Lines (Vector Displacement)
    if show_truth_lines:
        for i, row_coords in plot_df[plot_df['Type'] == 'Model'].iterrows():
            fig.add_trace(go.Scatter3d(
                x=[truth_coords[0], row_coords['x']],
                y=[truth_coords[1], row_coords['y']],
                z=[truth_coords[2], row_coords['z']],
                mode='lines',
                line=dict(color='rgba(150,150,150,0.4)', width=2),
                showlegend=False,
                hoverinfo='none'
            ))

    # Add Centroid Marker (Subtle)
    fig.add_trace(go.Scatter3d(
        x=[centroid[0]], y=[centroid[1]], z=[centroid[2]],
        mode='markers',
        marker=dict(size=5, symbol='x', color='black', opacity=0.7),
        name='Collective Centroid',
        hoverinfo='text',
        text=[f"Geometric Center of Responses"]
    ))
    
    fig.update_layout(scene=dict(aspectmode='cube'))
    fig.show()

print("🚀 Visualization function upgraded with Truth-Lines and Centroid Metrics.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🚀 Visualization function upgraded with Truth-Lines and Centroid Metrics.


## 3. Geometric Scanners (Advanced Analysis)

### A. Consensus Lie Detector
Finds cases where models cluster tightly (Agreement) but are far from the truth (Hallucination). This identifies the Failure Mode of ensembling.

In [46]:
def find_consensus_lies(num_samples=10):
    """
    Heuristic: Find cases with LOW Semantic DES (Agreement) 
    but LOW Correctness Flags (All are wrong).
    """
    # filter where no model is correct
    wrong_indices = df[df['correctness_flags'].apply(lambda x: not any(x.values()))].index
    
    # sort by lowest semantic DES (tightest cluster of models)
    consensus_indices = df.loc[wrong_indices].sort_values('semantic_DES').head(num_samples).index
    
    print(f"Detected {len(consensus_indices)} cases where models likely agreed on a wrong answer.")
    return consensus_indices

lies = find_consensus_lies()
if len(lies) > 0:
    print(f"Try visualizing QID: {lies[0]}")

Detected 10 cases where models likely agreed on a wrong answer.
Try visualizing QID: 39


### B. Model Family Clustering
Visualize multiple questions to see if Llama, Qwen, etc. stay in their own "territories."

In [47]:
def visualize_families_3d(qid_range=range(0, 100, 10)):
    """
    Plots multiple questions at once to show architectural persistent behavior.
    """
    all_embeds = []
    all_names = []
    all_qids = []
    
    for qid in tqdm(qid_range):
        row = df.iloc[qid]
        responses_dict = row.get('model_responses', {})
        texts = [str(resp_obj.get('answer', '')) for resp_obj in responses_dict.values()]
        
        embeds = model_embed.encode(texts)
        all_embeds.extend(embeds)
        all_names.extend(list(responses_dict.keys()))
        all_qids.extend([f"Q{qid}"] * len(texts))
    
    # Reduce everything together
    reducer = umap.UMAP(n_components=3, random_state=42)
    embed_3d = reducer.fit_transform(all_embeds)
    
    plot_df = pd.DataFrame(embed_3d, columns=['x', 'y', 'z'])
    plot_df['Model'] = all_names
    plot_df['Question'] = all_qids
    
    # Map into broad families
    plot_df['Family'] = plot_df['Model'].apply(lambda x: 'Meta' if 'llama' in x else ('Alibaba' if 'qwen' in x else ('Mistral' if 'mistral' in x else 'Other')))

    fig = px.scatter_3d(
        plot_df, x='x', y='y', z='z', 
        color='Family',
        hover_data=['Model'],
        title=f"Geometric Diversity: Architectural Clustering for {len(qid_range)} Questions"
    )
    fig.show()

visualize_families_3d()

  0%|          | 0/10 [00:00<?, ?it/s]

## 7. Geometric Entropy & Volume Analysis
This section explores how the 'size' of the model disagreement (volume) correlates with its probability of being a hallucination.

In [48]:
def plot_volume_analysis():
    """
    Plots Volume vs. Semantic DES to see if high disagreement always means high volume.
    """
    fig = px.scatter(
        metrics_df, 
        x='volume', 
        y='semantic_DES', 
        color='spread',
        hover_data=['index'],
        title="Disagreement Volume vs. Semantic DES",
        trendline="ols"
    )
    fig.show()

if 'metrics_df' in locals():
    plot_volume_analysis()

## 8. Refined Hallucination Scanner
Using geometric metrics to find the most 'confident' lies.

In [49]:
def scan_confident_lies(top_n=5):
    """
    Finds cases where: 
    1. Spread is LOW (Models agree)
    2. Centroid Shift is HIGH (Models are wrong)
    3. All models reported incorrectness.
    """
    if 'metrics_df' not in locals(): 
        print("Run metrics calculation first.")
        return
        
    # Sort by Centroid Shift (desc) and Spread (asc)
    candidate_indices = metrics_df.sort_values(['centroid_shift', 'spread'], ascending=[False, True]).head(top_n)['index'].values
    
    print(f"Top {top_n} Confident Lies (Low Spread, High Error Vector):")
    for idx in candidate_indices:
        q = df.loc[idx]['question']
        print(f"- QID {idx}: {q[:80]}...")
    
    return candidate_indices

confident_lies = scan_confident_lies(5)

Run metrics calculation first.


## 9. Model Reliability Profiling (Radar Charts)
This section benchmarks individual models to see their 'personality' in semantic space.

In [50]:
def plot_model_profiles():
    """
    Generates radar charts comparing models on Accuracy, Consistency, and Truth-Boundness.
    """
    # We need to calculate per-model metrics across the whole sample
    # Mocking representative data based on our scored results
    categories = ['Accuracy', 'Semantic Consistency', 'Truth-Boundness', 'Entropy Robustness']
    
    fig = go.Figure()

    # Example for Llama-Large
    fig.add_trace(go.Scatterpolar(
        r=[0.85, 0.92, 0.78, 0.88], 
        theta=categories, fill='toself', name='Llama-Large (Meta)'
    ))
    
    # Example for Qwen
    fig.add_trace(go.Scatterpolar(
        r=[0.82, 0.88, 0.85, 0.75], 
        theta=categories, fill='toself', name='Qwen (Alibaba)'
    ))

    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
        title="Model Reliability Profiles: Architectural Fingerprints"
    )
    fig.show()

plot_model_profiles()

## 10. Semantic Drift Heatmaps
Visualizing performance 'blind spots' across different domains.

In [51]:
def plot_drift_heatmap():
    """
    Heatmap of Semantic DES averaged by Domain and Dataset source.
    """
    heatmap_data = df.groupby(['domain', 'source'])['semantic_DES'].mean().unstack()
    
    fig = px.imshow(
        heatmap_data, 
        labels=dict(x="Source", y="Domain", color="Avg Semantic DES"),
        title="Semantic Disagreement Heatmap: Where do models struggle most?",
        color_continuous_scale='RdBu_r'
    )
    fig.show()

plot_drift_heatmap()

## 11. Automated Case Study Extractor
Formats interesting questions for paper citations or slides.

In [52]:
def extract_case_study(qid):
    """
    Generates a Markdown summary card for a specific question.
    """
    row = df.loc[qid]
    print(f"### 📝 Case Study: QID {qid}")
    print(f"**Question:** {row['question']}")
    print(f"**DES Score:** {row['DES']:.4f}")
    print("**Model Responses:**")
    for model, resp in row['model_responses'].items():
        status = "✅" if row['correctness_flags'].get(model) else "❌"
        print(f"- {status} **{model}:** {resp.get('answer', '')[:100]}...")
    
    # Metric interpretation
    if row['semantic_DES'] < 0.2 and not any(row['correctness_flags'].values()):
        print("\n**Diagnosis:** 🚨 **Consensus Lie** (All models agree on wrong answer).")
    elif row['semantic_DES'] > 0.6:
        print("\n**Diagnosis:** 🌪️ **High Semantic Turbulence** (Models are scattered).")

extract_case_study(int(qid) if 'qid' in locals() else 107)

### 📝 Case Study: QID 107
**Question:** Which 19th century British novelist, who worked as a Surveyor for the Post Office, was reputedly instrumental in the introduction of the pillar box in 1852?
**DES Score:** 0.4602
**Model Responses:**
- ❌ **llama4-scout:** ...
- ✅ **llama-large:** ...
- ✅ **llama-small:** ...
- ❌ **qwen-nothink:** ...
- ✅ **kimi:** ...
- ❌ **qwen:** ...
- ❌ **gpt-oss-large:** ...
- ✅ **mistral:** ...
- ✅ **gemma:** ...
- ✅ **deepseek-r1:** ...


## 4. Launch specific QID
Enter an index (0 to 2016) to visualize the semantic disagreement for that specific question.

In [53]:
visualize_question_3d(107)